In [1]:
require_relative '../modules/aoc'

puzzle = Puzzle.new(2016,24) do | input |
    input.chomp
end

puzzle.display(1)  
# puzzle.display(2)

"<article class=\"day-desc\"><h2>--- Day 24: Air Duct Spelunking ---</h2>\n<p>You've finally met your match; the doors that provide access to the roof are locked tight, and all of the controls and related electronics are inaccessible. You simply can't reach them.</p>\n<p>The robot that cleans the air ducts, however, <em>can</em>.</p>\n<p>It's not a very fast <span title=\"The Brave Little Air Duct Cleaning Robot That Could\">little robot</span>, but you reconfigure it to be able to interface with some of the exposed wires that have been routed through the <a href=\"https://en.wikipedia.org/wiki/HVAC\">HVAC</a> system. If you can direct it to each of those locations, you should be able to bypass the security controls.</p>\n<p>You extract the duct layout for this area from some blueprints you acquired and create a map with the relevant locations marked (your puzzle input). <code>0</code> is your current location, from which the cleaning robot embarks; the other numbers are (in <em>no particular order</em>) the locations the robot needs to visit at least once each. Walls are marked as <code>#</code>, and open passages are marked as <code>.</code>. Numbers behave like open passages.</p>\n<p>For example, suppose you have a map like the following:</p>\n<pre><code>###########\n#0.1.....2#\n#.#######.#\n#4.......3#\n###########\n</code></pre>\n<p>To reach all of the points of interest as quickly as possible, you would have the robot take the following path:</p>\n<ul>\n<li>\n<code>0</code> to <code>4</code> (<code>2</code> steps)</li>\n<li>\n<code>4</code> to <code>1</code> (<code>4</code> steps; it can't move diagonally)</li>\n<li>\n<code>1</code> to <code>2</code> (<code>6</code> steps)</li>\n<li>\n<code>2</code> to <code>3</code> (<code>2</code> steps)</li>\n</ul>\n<p>Since the robot isn't very fast, you need to find it the <em>shortest route</em>. This path is the fewest steps (in the above example, a total of <code>14</code>) required to start at <code>0</code> and then visit every other location at least once.</p>\n<p>Given your actual map, and starting from location <code>0</code>, what is the <em>fewest number of steps</em> required to visit every non-<code>0</code> number marked on the map at least once?</p>\n</article>"

## Approach

- Find the shortests distances between all the number pairs
- Calculate the total distances of all possible permutations of visiting the nodes (with 0 as the start point)
- Take the minimum

For part 2 add 0 after each permutation to return back


In [2]:
# A class to parse an AOC maze
# anything not equal to the wall or empty string is logged as a POI but is passable

class AOC::Maze
    def initialize(maze, wall, empty)
        lines = maze.lines
        @maze = Array.new(lines[0].length) { Array.new(lines.length)  }
        @poi = {}
        lines.each_with_index { | l, idx_y |  l.chomp.chars.each_with_index { | c, idx_x | 
                @maze[idx_x][idx_y] = case c
                when wall
                    false
                when empty
                    true
                else
                    @poi[c] = [idx_x, idx_y]
                    true
                end
            } 
        }
    end
    def [](idx)
        @maze[idx]
    end
    def poi(label)
        @poi[label]
    end
    def shortest(from, to)
        puts "Path from #{from} -> #{to}"
        visited = [from]
        paths = [[from]]
        while true
            newpaths = []
#             p paths
            paths.each do | p |
                tail = p[-1]
#                 p tail
                x, y = tail
                [[x-1,y],[x+1,y],[x,y+1],[x,y-1]].each do | neighbour |
                    return p + [neighbour] if neighbour == to
                    next if visited.include?(neighbour)
                    nx,ny = neighbour
                    next unless @maze[nx][ny]
#                     p neighbour
                    newpaths << p + [neighbour]
                    visited << neighbour
#                     p newpaths
                end    
            end
            raise "No path found" if newpaths == []
            paths = newpaths
        end
    end
end
test = <<EOF ###########
#0.1.....2#
#.#######.#
#4.......3#
###########
EOF
m = AOC::Maze.new(puzzle.input.chomp, "#", ".")
locations = (puzzle.input.scan /[0-9]+/)
p locations
p m.poi("0")
m.shortest(m.poi("0"), m.poi("4"))

["4", "6", "0", "1", "7", "3", "5", "2"]
[153, 9]
Path from [153, 9] -> [131, 1]


[[153, 9], [154, 9], [155, 9], [155, 10], [155, 11], [154, 11], [153, 11], [152, 11], [151, 11], [151, 10], [151, 9], [150, 9], [149, 9], [149, 10], [149, 11], [148, 11], [147, 11], [147, 12], [147, 13], [147, 14], [147, 15], [148, 15], [149, 15], [149, 16], [149, 17], [149, 18], [149, 19], [149, 20], [149, 21], [149, 22], [149, 23], [149, 24], [149, 25], [149, 26], [149, 27], [149, 28], [149, 29], [148, 29], [147, 29], [147, 30], [147, 31], [146, 31], [145, 31], [145, 32], [145, 33], [144, 33], [143, 33], [143, 32], [143, 31], [143, 30], [143, 29], [142, 29], [141, 29], [141, 28], [141, 27], [141, 26], [141, 25], [141, 24], [141, 23], [140, 23], [139, 23], [138, 23], [137, 23], [137, 22], [137, 21], [137, 20], [137, 19], [137, 18], [137, 17], [137, 16], [137, 15], [136, 15], [135, 15], [135, 14], [135, 13], [134, 13], [133, 13], [133, 12], [133, 11], [133, 10], [133, 9], [132, 9], [131, 9], [131, 8], [131, 7], [132, 7], [133, 7], [133, 6], [133, 5], [133, 4], [133, 3], [133, 2], [133,

In [4]:
# distance a->b == b->a so we need to calculate the different combinations
distances_to_calculate = locations.combination(2)
DISTANCES = {}
distances_to_calculate.each_with_object(DISTANCES) do | trip, dists | 
    from, to = trip
    l = m.shortest(m.poi(from),m.poi(to)).length - 1

    DISTANCES[[from,to]]= l
    DISTANCES[[to,from]] = l
end

(irb):2: warning: already initialized constant Object::DISTANCES
(irb):3: warning: previous definition of DISTANCES was here


Path from [131, 1] -> [23, 9]
Path from [131, 1] -> [153, 9]
Path from [131, 1] -> [167, 17]
Path from [131, 1] -> [7, 21]
Path from [131, 1] -> [171, 31]
Path from [131, 1] -> [1, 41]
Path from [131, 1] -> [139, 41]
Path from [23, 9] -> [153, 9]
Path from [23, 9] -> [167, 17]
Path from [23, 9] -> [7, 21]
Path from [23, 9] -> [171, 31]
Path from [23, 9] -> [1, 41]
Path from [23, 9] -> [139, 41]
Path from [153, 9] -> [167, 17]
Path from [153, 9] -> [7, 21]
Path from [153, 9] -> [171, 31]
Path from [153, 9] -> [1, 41]
Path from [153, 9] -> [139, 41]
Path from [167, 17] -> [7, 21]
Path from [167, 17] -> [171, 31]
Path from [167, 17] -> [1, 41]
Path from [167, 17] -> [139, 41]
Path from [7, 21] -> [171, 31]
Path from [7, 21] -> [1, 41]
Path from [7, 21] -> [139, 41]
Path from [171, 31] -> [1, 41]
Path from [171, 31] -> [139, 41]
Path from [1, 41] -> [139, 41]


{["4", "6"]=>172, ["6", "4"]=>172, ["4", "0"]=>94, ["0", "4"]=>94, ["4", "1"]=>108, ["1", "4"]=>108, ["4", "7"]=>184, ["7", "4"]=>184, ["4", "3"]=>94, ["3", "4"]=>94, ["4", "5"]=>198, ["5", "4"]=>198, ["4", "2"]=>60, ["2", "4"]=>60, ["6", "0"]=>230, ["0", "6"]=>230, ["6", "1"]=>244, ["1", "6"]=>244, ["6", "7"]=>60, ["7", "6"]=>60, ["6", "3"]=>230, ["3", "6"]=>230, ["6", "5"]=>82, ["5", "6"]=>82, ["6", "2"]=>188, ["2", "6"]=>188, ["0", "1"]=>30, ["1", "0"]=>30, ["0", "7"]=>242, ["7", "0"]=>242, ["0", "3"]=>48, ["3", "0"]=>48, ["0", "5"]=>256, ["5", "0"]=>256, ["0", "2"]=>58, ["2", "0"]=>58, ["1", "7"]=>256, ["7", "1"]=>256, ["1", "3"]=>30, ["3", "1"]=>30, ["1", "5"]=>270, ["5", "1"]=>270, ["1", "2"]=>68, ["2", "1"]=>68, ["7", "3"]=>242, ["3", "7"]=>242, ["7", "5"]=>50, ["5", "7"]=>50, ["7", "2"]=>200, ["2", "7"]=>200, ["3", "5"]=>256, ["5", "3"]=>256, ["3", "2"]=>54, ["2", "3"]=>54, ["5", "2"]=>214, ["2", "5"]=>214}

In [6]:
locations = (puzzle.input.scan /[0-9]+/)
p locations

DISTANCES = {["4", "6"]=>172, ["6", "4"]=>172, ["4", "0"]=>94, ["0", "4"]=>94, ["4", "1"]=>108, ["1", "4"]=>108, ["4", "7"]=>184, ["7", "4"]=>184, ["4", "3"]=>94, ["3", "4"]=>94, ["4", "5"]=>198, ["5", "4"]=>198, ["4", "2"]=>60, ["2", "4"]=>60, ["6", "0"]=>230, ["0", "6"]=>230, ["6", "1"]=>244, ["1", "6"]=>244, ["6", "7"]=>60, ["7", "6"]=>60, ["6", "3"]=>230, ["3", "6"]=>230, ["6", "5"]=>82, ["5", "6"]=>82, ["6", "2"]=>188, ["2", "6"]=>188, ["0", "1"]=>30, ["1", "0"]=>30, ["0", "7"]=>242, ["7", "0"]=>242, ["0", "3"]=>48, ["3", "0"]=>48, ["0", "5"]=>256, ["5", "0"]=>256, ["0", "2"]=>58, ["2", "0"]=>58, ["1", "7"]=>256, ["7", "1"]=>256, ["1", "3"]=>30, ["3", "1"]=>30, ["1", "5"]=>270, ["5", "1"]=>270, ["1", "2"]=>68, ["2", "1"]=>68, ["7", "3"]=>242, ["3", "7"]=>242, ["7", "5"]=>50, ["5", "7"]=>50, ["7", "2"]=>200, ["2", "7"]=>200, ["3", "5"]=>256, ["5", "3"]=>256, ["3", "2"]=>54, ["2", "3"]=>54, ["5", "2"]=>214, ["2", "5"]=>214}

["4", "6", "0", "1", "7", "3", "5", "2"]


(irb):3: warning: already initialized constant Object::DISTANCES
(irb):2: warning: previous definition of DISTANCES was here


{["4", "6"]=>172, ["6", "4"]=>172, ["4", "0"]=>94, ["0", "4"]=>94, ["4", "1"]=>108, ["1", "4"]=>108, ["4", "7"]=>184, ["7", "4"]=>184, ["4", "3"]=>94, ["3", "4"]=>94, ["4", "5"]=>198, ["5", "4"]=>198, ["4", "2"]=>60, ["2", "4"]=>60, ["6", "0"]=>230, ["0", "6"]=>230, ["6", "1"]=>244, ["1", "6"]=>244, ["6", "7"]=>60, ["7", "6"]=>60, ["6", "3"]=>230, ["3", "6"]=>230, ["6", "5"]=>82, ["5", "6"]=>82, ["6", "2"]=>188, ["2", "6"]=>188, ["0", "1"]=>30, ["1", "0"]=>30, ["0", "7"]=>242, ["7", "0"]=>242, ["0", "3"]=>48, ["3", "0"]=>48, ["0", "5"]=>256, ["5", "0"]=>256, ["0", "2"]=>58, ["2", "0"]=>58, ["1", "7"]=>256, ["7", "1"]=>256, ["1", "3"]=>30, ["3", "1"]=>30, ["1", "5"]=>270, ["5", "1"]=>270, ["1", "2"]=>68, ["2", "1"]=>68, ["7", "3"]=>242, ["3", "7"]=>242, ["7", "5"]=>50, ["5", "7"]=>50, ["7", "2"]=>200, ["2", "7"]=>200, ["3", "5"]=>256, ["5", "3"]=>256, ["3", "2"]=>54, ["2", "3"]=>54, ["5", "2"]=>214, ["2", "5"]=>214}

In [7]:
DISTANCES

{["4", "6"]=>172, ["6", "4"]=>172, ["4", "0"]=>94, ["0", "4"]=>94, ["4", "1"]=>108, ["1", "4"]=>108, ["4", "7"]=>184, ["7", "4"]=>184, ["4", "3"]=>94, ["3", "4"]=>94, ["4", "5"]=>198, ["5", "4"]=>198, ["4", "2"]=>60, ["2", "4"]=>60, ["6", "0"]=>230, ["0", "6"]=>230, ["6", "1"]=>244, ["1", "6"]=>244, ["6", "7"]=>60, ["7", "6"]=>60, ["6", "3"]=>230, ["3", "6"]=>230, ["6", "5"]=>82, ["5", "6"]=>82, ["6", "2"]=>188, ["2", "6"]=>188, ["0", "1"]=>30, ["1", "0"]=>30, ["0", "7"]=>242, ["7", "0"]=>242, ["0", "3"]=>48, ["3", "0"]=>48, ["0", "5"]=>256, ["5", "0"]=>256, ["0", "2"]=>58, ["2", "0"]=>58, ["1", "7"]=>256, ["7", "1"]=>256, ["1", "3"]=>30, ["3", "1"]=>30, ["1", "5"]=>270, ["5", "1"]=>270, ["1", "2"]=>68, ["2", "1"]=>68, ["7", "3"]=>242, ["3", "7"]=>242, ["7", "5"]=>50, ["5", "7"]=>50, ["7", "2"]=>200, ["2", "7"]=>200, ["3", "5"]=>256, ["5", "3"]=>256, ["3", "2"]=>54, ["2", "3"]=>54, ["5", "2"]=>214, ["2", "5"]=>214}

In [8]:
locations.delete("0")
locations

paths1 = locations.permutation.map { | p | ["0"] + p }.freeze
paths2 = paths1.map { | p | p + ["0"]}.freeze
# p paths1
def distance(from,to)
        DISTANCES[[from,to]]
end

PART1 = (paths1.map do | p |
 #    p p
    p.each_cons(2).reduce(0) do | acc, trip |
#         p trip
        from, to = trip
#          p "Calculate distance #{from} -> #{to}"
        dist = distance(from,to)
#         p dist
        acc += dist 
    end
end.min)

PART2 = (paths2.map do | p |
#     p p
    p.each_cons(2).map do | trip |
#         p trip
        from, to = trip
#         p "Calculate distance #{from} -> #{to}"
        distance(from,to)
    end.sum
end.min)
nil

In [9]:
p PART1
p PART2

456
704


704

In [12]:
def puzzle.part1
    PART1
end

def puzzle.part2
    PART2
end
puzzle.solve

Part1:	456	  0.000118   0.000015   0.000133 (  0.000128)
Part2:	704	  0.000105   0.000040   0.000145 (  0.000103)


In [2]:
#!/usr/bin/env ruby

require 'set'

file_path = File.expand_path("./input/24.txt")
input     = File.read(file_path)

maze = input.split("\n").map { |line| line.chars }

targets = {}

maze.each_with_index do |row, y|
  row.each_with_index do |cell, x|
    targets[cell] = [x, y] if cell =~ /\d/
  end
end

distances = Hash.new { |h, k| h[k] = {} }
p distances
(targets.to_a).product(targets.to_a).each do |(s, (sx, sy)), (g, (gx, gy))|
  next if s == g || distances[s][g]

  visited = Set.new
  queue = [[sx, sy, 0]]

  visited.add(queue.first[0..1])
  reached_goal = false

  while !reached_goal
    current = queue.shift

    if current[0..1] == [gx, gy]
      distances[s][g] = current[2]
      distances[g][s] = current[2]

      reached_goal = true
    end

    x = current[0]
    y = current[1]
    step = current[2]

    [[x+1, y], [x-1, y], [x, y+1], [x, y-1]].each do |x, y|
      next if x < 0 || y < 0
      next if maze[y][x] == "#"
      next if visited.include?([x, y])

      visited.add([x, y])
      new_location = [x, y, step + 1]
      queue << new_location
    end
  end
end

lowest = 1_000

(1..7).map(&:to_s).permutation.each do |sequence|
  sequence.unshift("0")

  sum = sequence.each_cons(2).
    map { |start, goal| distances[start][goal] }.
    inject(:+)

  lowest = [lowest, sum].min
end

puts lowest
lowest = 1_000
(1..7).map(&:to_s).permutation.each do |sequence|
  sequence.unshift("0")
  sequence = sequence.append("0")
  # p sequence
  sum = sequence.each_cons(2).
    map { |start, goal| distances[start][goal] }.
    inject(:+)

  lowest = [lowest, sum].min
end

puts lowest

{}
456
704
